# LIGER best - nph_cerebellum

Seeds: `1-10`

Metrics: BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR


In [1]:
import gc
import importlib.util
import os
import sys

os.environ["PYTHONNOUSERSITE"] = "1"
sys.path = [
    path
    for path in sys.path
    if ".local/lib/python3.10/site-packages" not in path
]

PROJECT_ROOT = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics'
os.environ["R_HOME"] = f"{PROJECT_ROOT}/CondaEnv/pyliger_env/lib/R"
sys.path.insert(0, f"{PROJECT_ROOT}/scib/")

import anndata as ad
import numpy as np
import pandas as pd

OPTIMIZER_PATH = (
    f"{PROJECT_ROOT}/scripts/optuna_multi_metrics/"
    "optuna_liger/optimize_liger_multi_metrics.py"
)
optimizer_spec = importlib.util.spec_from_file_location(
    "liger_opt",
    OPTIMIZER_PATH,
)
if optimizer_spec is None or optimizer_spec.loader is None:
    raise ImportError(f"Cannot load LIGER optimizer: {OPTIMIZER_PATH}")
liger_opt = importlib.util.module_from_spec(optimizer_spec)
sys.modules[optimizer_spec.name] = liger_opt
optimizer_spec.loader.exec_module(liger_opt)


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET_NAME = 'nph_cerebellum'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEY = 'Sample'
LABEL_KEY = 'Celltype'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

SEEDS = tuple(range(1, 11))
MODE = "core"
LAYER = "counts"
N_NEIGHBORS = 15
EMBED_KEY = "X_liger"

liger_opt.DATASET_NAME = DATASET_NAME
liger_opt.DATA_PATH = DATA_PATH
liger_opt.BATCH_KEY = BATCH_KEY
liger_opt.LABEL_KEY = LABEL_KEY
liger_opt.MODE = MODE
liger_opt.LAYER = LAYER
liger_opt.USE_BAMI = "BAMI" in SELECTED_METRICS
liger_opt.USE_CASW = "CASW" in SELECTED_METRICS
liger_opt.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
liger_opt.USE_TRAJ = "TRAJ" in SELECTED_METRICS
liger_opt.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
liger_opt.USE_BRAS = "BRAS" in SELECTED_METRICS
liger_opt.USE_KBET = "KBET" in SELECTED_METRICS
liger_opt.USE_CPCR = "CPCR" in SELECTED_METRICS

assert DATASET_NAME == 'nph_cerebellum'
assert BATCH_KEY == 'Sample'
assert LABEL_KEY == 'Celltype'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'BRAS', 'KBET', 'CPCR')
assert SEEDS == tuple(range(1, 11))
assert liger_opt.CORE_MAX_ITERS == 30
assert liger_opt.N_NEIGHBORS == N_NEIGHBORS
assert liger_opt.EMBED_KEY == EMBED_KEY

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_key": BATCH_KEY,
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(SEEDS),
        "mode": MODE,
        "layer": LAYER,
    },
    name="configuration",
)


dataset                                         nph_cerebellum
data_path    /ubc/cs/research/beaver/projects/Single-cell-i...
batch_key                                               Sample
label_key                                             Celltype
metrics               [BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR]
seeds                          [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
mode                                                      core
layer                                                   counts
Name: configuration, dtype: object

In [3]:
liger_opt.validate_metric_dependencies()
liger_opt.validate_configuration(list(SELECTED_METRICS))
liger_opt.validate_dataset_contents()


Dataset preflight: n_obs=84619 n_batches=2 n_labels=8
CPCR preflight: eligible_labels=8


In [4]:
OPTUNA_STUDY_NAME = 'liger_nph_cerebellum_joint_expanded_metrics_scgraphm_v1'
OPTUNA_BEST_TRIAL = 15
OPTUNA_BEST_VALUE = 0.7457946872944343
OPTUNA_BEST_PARAMS = {'k': 32, 'value_lambda': 0.5033965087804688}

LIGER_BEST_ARGS = {
    "k": OPTUNA_BEST_PARAMS["k"],
    "value_lambda": OPTUNA_BEST_PARAMS["value_lambda"],
    "max_iters": 30,
    "mode": "core",
    "layer": "counts",
}


def run_liger_best(adata, seed):
    return liger_opt.run_liger(
        adata=adata,
        k=LIGER_BEST_ARGS["k"],
        lam=LIGER_BEST_ARGS["value_lambda"],
        max_iterations=LIGER_BEST_ARGS["max_iters"],
        mode=LIGER_BEST_ARGS["mode"],
        layer=LIGER_BEST_ARGS["layer"],
        seed=seed,
    )


pd.Series(
    {
        "study": OPTUNA_STUDY_NAME,
        "best_trial": OPTUNA_BEST_TRIAL,
        "best_value": OPTUNA_BEST_VALUE,
        **LIGER_BEST_ARGS,
    },
    name="best_setting",
)


study           liger_nph_cerebellum_joint_expanded_metrics_sc...
best_trial                                                     15
best_value                                               0.745795
k                                                              32
value_lambda                                             0.503397
max_iters                                                      30
mode                                                         core
layer                                                      counts
Name: best_setting, dtype: object

In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    return liger_opt.compute_joint_metrics(adata_raw, adata_int)


In [6]:
results = []

for seed in SEEDS:
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_liger_best(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()

results_df = pd.DataFrame(results)
results_df



===== seed 1 =====


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the

Removing 1023 genes not expressing in FM3.


Removing 246 genes not expressing in MM1.


[LIGER PARAMS] function=optimize_ALS k=32 value_lambda=0.5033965087804688 max_iters=30 rand_seed=1


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.7188453322386544


optimised clustering against liger_metric_label
optimal cluster resolution: 0.2
optimal score: 0.8233232493906156
BRAS...


seed           10.000000
BAMI            0.823323
CASW            0.829614
SCGRAPHM        0.374959
BRAS            0.844074
KBET            0.617056
CPCR            0.748374
bio_score       0.675965
batch_score     0.736501
joint_score     0.700180
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,1,0.797208,0.827366,0.548570,0.866577,0.678643,0.788522,0.724382,0.777914,0.745795
1,2,0.770969,0.829976,0.390658,0.838954,0.631973,0.685803,0.663868,0.718910,0.685885
2,3,0.783552,0.820874,0.341725,0.853937,0.645443,0.893009,0.648717,0.797463,0.708215
3,4,0.811502,0.816268,0.361548,0.843845,0.623282,0.868765,0.663106,0.778630,0.709316
4,5,0.763797,0.826850,0.336912,0.855552,0.662143,0.882671,0.642520,0.800122,0.705560
5,6,0.761265,0.813052,0.345554,0.875904,0.659709,0.898456,0.639957,0.811356,0.708517
6,7,0.807808,0.835504,0.464404,0.851466,0.620421,0.853218,0.702572,0.775035,0.731557
7,8,0.790644,0.806064,0.361776,0.843803,0.642227,0.652577,0.652828,0.712869,0.676844
8,9,0.745556,0.837173,0.375268,0.769553,0.590061,0.889916,0.652666,0.749843,0.691537
9,10,0.823323,0.829614,0.374959,0.844074,0.617056,0.748374,0.675965,0.736501,0.700180


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.785563,0.023742,0.745556,0.823323
CASW,0.824274,0.009502,0.806064,0.837173
SCGRAPHM,0.390138,0.063130,0.336912,0.548570
BRAS,0.844366,0.027170,0.769553,0.875904
KBET,0.637096,0.024637,0.590061,0.678643
CPCR,0.816131,0.087060,0.652577,0.898456
bio_score,0.666658,0.025949,0.639957,0.724382
batch_score,0.765864,0.032811,0.712869,0.811356
joint_score,0.706341,0.019380,0.676844,0.745795
